<a href="https://colab.research.google.com/github/NehalShahu/Gen_AI/blob/main/GenAI_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets -q

import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


data = {
    "text": [
        "India won the cricket match by 5 wickets.",
        "The football team reached the final.",
        "The player scored two goals in the match.",
        "The cricket tournament will start next week.",

        "The government announced new economic policies.",
        "The president met with political leaders.",
        "Parliament passed the new education bill.",
        "The election campaign started across the country.",

        "Apple launched a new smartphone with advanced technology.",
        "Artificial intelligence is changing the technology industry.",
        "The company developed a new computer processor.",
        "Researchers created a powerful machine learning model."
    ],

    "label": [
        0, 0, 0, 0,       # Sports
        1, 1, 1, 1,       # Politics
        2, 2, 2, 2        # Technology
    ]
}

df = pd.DataFrame(data)

print("Dataset:")
print(df)



dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(
    test_size=0.25,
    seed=42
)


model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)



def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)


model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

id2label = {
    0: "Sports",
    1: "Politics",
    2: "Technology"
}

model.config.id2label = id2label



def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    accuracy = np.mean(predictions == labels)

    return {
        "accuracy": accuracy
    }


training_args = TrainingArguments(
    output_dir="./news_classifier",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=1,
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

print("\nTraining started...\n")

trainer.train()

print("\nTraining completed!")



results = trainer.evaluate()

print("\nEvaluation Results:")
print(results)


def classify_news(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    predicted_class = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    confidence = probabilities[0][predicted_class].item()

    print("\nNews Article:")
    print(text)

    print("Category:", id2label[predicted_class])
    print("Confidence:", round(confidence * 100, 2), "%")




classify_news(
    "The Indian cricket team won the match after an excellent performance."
)

classify_news(
    "The government announced new policies before the upcoming election."
)

classify_news(
    "Scientists developed a new artificial intelligence system."
)

Dataset:
                                                 text  label
0           India won the cricket match by 5 wickets.      0
1                The football team reached the final.      0
2           The player scored two goals in the match.      0
3        The cricket tournament will start next week.      0
4     The government announced new economic policies.      1
5           The president met with political leaders.      1
6           Parliament passed the new education bill.      1
7   The election campaign started across the country.      1
8   Apple launched a new smartphone with advanced ...      2
9   Artificial intelligence is changing the techno...      2
10    The company developed a new computer processor.      2
11  Researchers created a powerful machine learnin...      2


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy
1,1.080437,1.097270,0.666667
2,1.111605,1.089310,0.333333
3,0.960886,1.074685,0.333333
4,0.853210,1.065103,0.666667
5,0.880681,1.061958,0.666667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed!


Training Loss,Validation Loss,Epoch,Accuracy
0.880681,1.061958,5,0.666667



Evaluation Results:
{'eval_loss': 1.0619579553604126, 'eval_accuracy': 0.6666666666666666}

News Article:
The Indian cricket team won the match after an excellent performance.
Category: Sports
Confidence: 36.38 %

News Article:
The government announced new policies before the upcoming election.
Category: Politics
Confidence: 35.11 %

News Article:
Scientists developed a new artificial intelligence system.
Category: Technology
Confidence: 45.41 %
